# Chapter 8

In this chapter we will look into **source estimation/reconstruction/localisation**. **Source Estimation** is the process of using the electrical signals measured on the scalp (2D) to figure out exactly where in the brain (3D) those signals came from.
<br />
> **The "Muffled House" Analogy:**
> 
> Imagine there is s party inside a house with thick concrete walls.
>   1. *The Source:* A drummer is playing in the living room.
>   2. *The Sensors:* You put microphones on the outside of the house walls.
>   3. *The Problem:* The concrete walls smear the sound. By the time the sound reaches the outside, it doesn't sound like it's coming from one sharp point anymore. It sounds like a vague "hum" coming from the whole side of the house.
>
> **Source Estimation** is the mathematical detective work of saying:
>
> *"Okay, the microphones on the North Wall are vibrating slightly more than the South Wall. Even though the sound is blurry, the math says there is a 90% chance the drummer is in the Living room and not the Kitchen."*
>   + ***Forward Model:** Knowing the phyics of the house (how thick the walls are).*
>   + ***Inverse Solution:** Using that knowledge to pinpoint the drummer.*

To perform *Source Estimation*, we need to combine our *Data* (what we measured) with *Anatomy* (the physical head). The diagram below illustrates the entire MNE pipeline.

<div style="text-align: center;">
<img src="./imgs/mne-python_flow_diagram.svg" alt="Workflow of the MNE software
" width="500"/>
<br />
Workflow of the MNE software.
<br />
*Source: <a href="https://mne.tools/stable/documentation/cookbook.html">MNE-Python website</a>
</div>

+ **The Data Pipeline (The "Blue" boxes ):** We have already completed the left-hand side of the diagram, except the **Noise Covariance** step (which is more of a final calculation step). This was all about processing the electrical signals:
    1. **Raw Data (`mne.io.Raw`):** We loaded the continuous recording from the EEG/MEG machine.
    2. **Preprocessing Data:** We filtered the data (1-40Hz) and removed artifacts (*ICA/SSP*) to clean up blinks and heartbeats.
    3. **Epoched Data (`mne.Epochs`):** We chopped the continuous signal into specific trails (e.g., -0.2s to 0.5s around the stimulus).
    4. **Averaged Data (`mne.Evoked`):** We averaged the trails to isolate the stable brain response (ERP) from the noise.
+ **Building the "Source Model" (The Red Box):** Now, to find *where* the signal comes from, we need to build a phyical model of the head. In this chapter we will cover the steps that needs processing as the intial few steps/data here already comes along the raw data:
    1. **Structural MRI (T1):** MRI scan of the subject's head to get the actual 3D shape of the brain and skull.
    2. **FreeSurfer Surfaces:** We use software to reconstruct the 3D surface of the brain (cortex) from the MRI.
    3. **BEM (Boundary Element Model):** The "Physics Engine". The model calculates how electricity conducts through the different layers (Brain $\to$ Skull $\to$ Scalp). Since the skill is an insulator, this step is critical for accuracy.
    4. **Co-registration (Head-MRI Trans):** Aligning the MRI (Anotomy) with the EEG sensors. This ensures the 3D head model sits in the exact same position as the sensors were during the recording.
    5. **Source Space (`src`):** We define a "Search Grid" by placing thousands of candidate dipoles all over the brain surface. These are the possible locations we will test.
+ **The Final Calculation (The Inverse Solution):** Once we the *Data* and the *Model*, we combine them:
    1. **Forward Solution:** The *"Cheat Sheet"* - it calculates: *"If a source fired at Position X, what pattern would the sensors see?"* - It does this for every singe  candidate source/dipole.
    2. **Noise Covariance:** A measure of the background noise in the sensors.
    3. **Inverse Operator:** The *"Solver"* - it combines the *Forward Solution* + *Noise Covariance* + *Evoked Data* to mathematically deduce the most likely source of the signal.
    4. **Source Estimatee (`stc`):** The final output - *a 3D movie showing brain activity on the cortex over time.*
> **Group Analysis**:
>
> The diagram ends at *Source estimate* (i.e., for one person). But usually, you have 20 subjects and in that case the final step for scientific studies is usually **Group Analysis**:
>   + **Morphing (`mne.compute_source_morph`) - Standardizing Brain:** We *"morph" (stretch/shrink)* everyone's brain activity onto a standard template brain (usually called `fsaverage`), because could be that subject A has a big head; subject B has a small head. You can't average them direclty.
>   + **Group Statistics (The final scientific conclusion):** Once everyone is on the sample template brain, we average them together to say: *"On average, humans activate the Auditory Cortex 100ms after a beep."*


## Libraries and Config
>**Note on Plotting Backends:**
> In this chapter, we are switching our plotting backend from `qt`/`QtAgg` and `pyvistaqt` to notebook. This ensures that the 3D interactive plots (Source Estimation, Co-registration) render correctly directly inside the Jupyter Notebook without crashing the kernel.
> ```python  
> %matplotlib qt
>
> matplotlib.use("QtAgg")
>
># comment this line of code below or replace 'notebook' with 'pyvistaqt'. As by default it is MNE 3D backend is set to 'pyvistaqt'.
> # mne.viz.set_3d_backend("notebook") 
> ```

In [ ]:
# This line is used in Jupyter notebooks to enable interactive plots with Matplotlib.
%matplotlib notebook

In [ ]:
import pathlib
import matplotlib
import matplotlib.pyplot as plt
import mne_bids
import mne

# Set matplotlib and MNE 3D backends to 'notebook' for interactive plots
matplotlib.use("notebook")
mne.viz.set_3d_backend("notebook")

mne.set_log_level("warning")

## Start with some fresh epochs

We will reload the raw data and create fresh epochs here. Ideally, we could continue with the variables from the previous chapter, but reloading ensures our data is in a known, clean state. This prevents any specific *filters*, *cropping*, or *modifications from the Time-Frequency analysis* ([Chapter 7](/chapter_7.ipynb)) from accidentally affecting our Source Estimation results.

> **Pro-tip: Don't "Apply" Projectors Manually**
> When performing *source estimation*, it is often best to leave your SSP projectors as inactive (`active=False`) in your `epochs` object.
>
>
> + *Why? The Inverse Operator* (`make_inverse_operator`) is smart.* It sees the inactive projectors in `epochs.info['projs']` and integrates them directly into the mathematical solution.
>
>
> + *The Risk: If you manually run `epochs.apply_proj()`, you permanently change the data.* If you then feed this to a function that also tries to account for the projectors (or if there's a mismatch with the *noise covariance*), you can run into numerical rank errors.

In [ ]:
bids_root = pathlib.Path(
    'out_data/sample_BIDS'
)

bids_path = mne_bids.BIDSPath(
    subject='01',
    session='01',
    task='audiovisual',
    run='01',
    datatype='meg',
    root=bids_root
)

raw_bids = mne_bids.read_raw_bids(bids_path=bids_path)

In [ ]:
raw_bids.load_data()
# band-pass filter the data
raw_bids.filter(
    l_freq=1.0, # low cut-off frequency in Hz
    h_freq=40.0 # high cut-off frequency in Hz
)

In [ ]:
events, event_id = mne.events_from_annotations(raw_bids)

# epochs timeline
tmin = -0.2  # start of each epoch (200ms before the trigger)
tmax = 0.5   # end of each epoch (500ms after the trigger)
baseline = (None, 0) # baseline correction period (from the first instant to t=0)

epochs = mne.Epochs(
    raw=raw_bids,
    events=events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    preload=True,
    proj=False  # do not apply SSP projections yet
)

epochs.save(
    pathlib.Path('out_data') / 'epochs_source_estimation-epo.fif', overwrite=True
)


## Loading and Visualizing the BEM
Fortunately, our sample dataset comes with a pre-computed **Boundary Element Model (BEM)**. As we discussed, this model defines the conductive compartments of the head (Inner Skull, Outer Skull, and Scalp) so the algorithm knows how electricity flows (and how it is blocked by the skull).

We should first visually inspect it to ensure the layers are accurate and topologically correct (i.e., they should not intersect). A common error is the brain surface poking through the inner skull, which would break the physics simulation.

We can call `mne.viz.plot_bem()` to overlay these contours (Red/Yellow lines) on top of the raw MRI (T1) slices.

> Note: These surfaces were produced using the FreeSurfer software. If you need to create them yourself from your own MRI scans, MNE provides wrapper functions:
> + `mne.bem.make_watershed_bem()`: Robust and works with standard T1 images.
> + `mne.bem.make_flash_bem()`: Higher precision but requires specific FLASH MRI sequences.

In [ ]:
sample_data_path = mne.datasets.sample.data_path()
subjects_dir = pathlib.Path(sample_data_path) / 'subjects'

In [ ]:
mne.viz.plot_bem(
    subject='sample',
    subjects_dir=subjects_dir,
    brain_surfaces='white', # Overlays the 'white matter' surface to check for intersections
    orientation='coronal'   # View slices from the front (Coronal plane)
)

## Co-registration (Head-MRI Trans)
As discussed, MNE needs to know exactly how the sensors (which were in the lab) align with the MRI/3D Model (which was taken in a scanner). This alignment process is called **Co-registration**.

We rely on three specific *"Anchor Points" (**Fiducials**)* that are identifiable in both the real world and the MRI:

1. *Nasion (NAS):* The bridge of the nose.
2. *Left Pre-auricular (LPA):* Just in front of the left ear canal.
3. *Right Pre-auricular (RPA):* Just in front of the right ear canal.

Luckily, MNE provides a GUI (`mne.gui.coregistration()`) to visualize and perfect this fit.

Steps in the GUI:
1. *Lock Fiducials::* Under the `MRI Fiducials` tab, check `Lock fiducials`. This prevents you from accidentally moving the anchor points on the MRI image while rotating view.
2. *Coarse Alignment:* Under `Translation (t) and Rotation (r)`, click `Fit fiducials`. This roughly snaps the EEG head shape into the MRI using just 3 anchor point.
3. *Fine Tuning (ICP):* Click `FIT ICP` (*Iterative Closest Point*). This is an algorithm that takes all the extra digitization points (hundreds of dots traced over the subject's scalp) and "morphs/rotates" the solution until the head shape perfectly matches the scalp surface. This is much more accurate tha  using just 3 points.
4. *Save:* Click the `Save` button (usually under `HEAD <> MRI Transform` tab) to save the transformation matrix (`-trans.fif`). This file tells MNE mathematically how to rotate the MRI to match the sensors.

> Note:
> + Montage: Ensure your epochs file has a montage/digitization points loaded! If you don't have digitization points (the cloud of dots), `Fit ICP` won't work.

In [ ]:
se_epochs_path = pathlib.Path('out_data') / 'epochs_source_estimation-epo.fif'

mne.gui.coregistration(
    subject='sample',
    subjects_dir=subjects_dir, # specify where the subject's MRI/BEM data is stored
    inst=se_epochs_path
)

### Visualizing the Alignment

After saving the transformation matrix (`-trans.fif`), we should verify that it works programmatically. We don't need the GUI for this; we can use `mne.viz.plot_alignment()`.

This function loads the "Head" (Sensors + Digitization points) and the "MRI" (Brain/Scalp surfaces) and uses your saved *transformation file* to overlay them. You should see the dots (digitization points) sitting perfectly on the skin surface.

In [ ]:
sample_data_path = mne.datasets.sample.data_path()
subjects_dir = pathlib.Path(sample_data_path) / 'subjects'
se_epochs_path = pathlib.Path('out_data') / 'epochs_source_estimation-epo.fif'

# This is the file you essentially "Saved" in the step above
# (Make sure this filename matches what you typed in the GUI!)
trans_fname = pathlib.Path('out_data') / 'sample_new-trans.fif'

info = mne.io.read_info(se_epochs_path) # read the info from the epochs file to know sensor locations

mne.viz.plot_alignment(
    info=info, 
    trans=trans_fname, # transformation file created with the GUI
    subject='sample', # specify which subject to use
    subjects_dir=subjects_dir,  # where the subject's MRI/BEM data is stored
    dig=True,
    verbose=True
)

## Compute the Source Space

As discussed, the **Source Space** defines our "search grid". We define thousands of specific points (*dipoles*) on the surface of the *cortex (white matter)*. During the analysis, the algorithm will check these specific points to see if the signal originated from there.

+ *Trade-off:* A dense grid means higher spatial precision, but much higher computational cost.

To set up a source space, we call `mne.setup_source_space()`. The key parameter is `spacing`, which defines the resolution:
+ `spacing='oct6'`: (Standard) Creates ~4,096 sources per hemisphere (8,192 total). This is the standard for publishable scientific results (roughly 4.9mm spacing).
+ `spacing='oct4'`: (Coarse) Creates ~1,026 sources per hermisphere. This is great for quick tutorials or debugging because it computes much faster.

In [ ]:
source_space = mne.setup_source_space(
    subject='sample',
    spacing='oct4',   # We use 'oct4' (coarse) for speed in this tutorial. Use 'oct6' for real analysis!
    subjects_dir=subjects_dir,
    add_dist=False    # 'False' saves time by skipping patch distance calculation. Set 'True' for real analysis.
)

print(source_space)

### Visualising the Grid

The source space object contains two parts (Left Hemishpere and Right Hemisphere). We can visualise these potential sources (pink dots) sitting on the white matter surface using `mne.viz.plot_alignment()`.

In [ ]:
mne.viz.plot_alignment(
    info=info,
    trans=trans_fname,
    subject='sample',
    src=source_space,     # Visualizes the source points (pink dots)
    subjects_dir=subjects_dir,
    dig=True,
    surfaces=['head-dense', 'white'], # Show the Scalp ('head-dense') and the Brain ('white')
    coord_frame='meg'     # View everything relative to the MEG helmet
)

## Compute the Forward Solution
As discussed, the **Forward Solution** answers the hypothetical question:

*"If a neuron fired at this exact spot on the grid, what would the sensors see?"*

We place a simulated electric dipole at every single vertex in our Source Space and calculate how its field propagates through the head tissues to reach the sensors. It is called "Forward" because we start with the Source (known) and calculate the Data (unknown).

In MNE, calculating the forward solution is a three-step process. First, we need a mathematical model of the head's tissues.

### 1. Create the BEM Conductivity Model

We use `mne.make_bem_model()` to define the conductivity of the head layers. This depends on whether we are analyzing MEG or EEG data:

1. *MEG (1-Layer Model): Magnetic fields pass through the skull and skin almost undistorted. We only need 1 layer (Inner Skull) to define the boundary of the brain.

    + *Default conductivity: `(0.3,)`*

2. *EEG (3-Layer Model):* Electric fields are strongly resisted by the skull. We need 3 layers (Inner Skull, Outer Skull, Scalp) to model this distortion accurately.

    + *Default conductivity: `(0.3, 0.006, 0.3)` representing (Brain, Skull, Skin).*

We also define `ico=4` (Icosahedron order 4), which creates a clean, decimated mesh of the surface triangles to make the physics calculations faster without losing much accuracy.

In [ ]:
# MEG only needs 1 layer (Inner Skull) because magnetic fields pass through skull/skin
conductivity_meg = (0.3,) 

# For EEG, we would use 3 layers:
# conductivity_eeg = (0.3, 0.006, 0.3)

bem_conductivity_model = mne.make_bem_model(
    subject='sample',
    ico=4,  # 4th order icosahedron: Creates a mesh of uniform triangles for the physics engine
    conductivity=conductivity_meg,
    subjects_dir=subjects_dir
)

bem_conductivity_model

### 2. Compute the BEM Solution
Now that we have defined the geometry and conductivity (`bem_conductivity_model`), we need to "solve" the physics equations.

We call `mne.make_bem_solution()`, which takes the model and pre-computes the mathematical matrices needed to simulate field propagation. This step essentially turns our geometric shapes into a linear algebra operator that the forward solver can use instantly.

In [ ]:
bem_solution = mne.make_bem_solution(bem_conductivity_model)

bem_solution

In [ ]:
# save the bem solution

bem_sol_fname = pathlib.Path('out_data') / 'sample_bem_solution-meg.fif'
mne.write_bem_solution(
    fname=bem_sol_fname,
    bem=bem_solution,
    overwrite=True
)

### 3. Creating the Forward Solution

Finally, we combine everything (Data Info + Head Alignment + Source Grid + Physics Model) to create the Forward Solution. This is the master "dictionary" that links brain locations to sensor values.

We call `mne.make_forward_solution()` with these parameters:

1. `info:` The sensor locations (from our epochs/raw file).
2. `trans:` The alignment file (`-trans.fif`) connecting the MRI head to the sensors.
3. `src:` The grid of candidate dipoles (`source_space`).
4. `bem:` The physics solution (`bem_sol`).
5. `meg=True, eeg=False:` *Crucial: We must enable only the sensors supported by our BEM. Since we built a 1-layer BEM (MEG-only), we must disable EEG or MNE will crash.*
6. `mindist=5.0`: A safety filter. It excludes any source that is closer than 5mm to the skull surface. Sources too close to the skull are mathematically unstable and likely noise.

In [ ]:
fwd_solution = mne.make_forward_solution(
    info=info,
    trans=trans_fname,
    src=source_space,
    bem=bem_solution,
    meg=True,
    eeg=False,
    mindist=5.0,  # Exclude sources < 5mm from inner skull
    n_jobs=-1     # Use all available CPU cores
)

fwd_solution

You might notice the output shows fewer vertices (e.g., `~474`) than our original Source Space (`~1000`).
This is because `mindist=5.0` removed the vertices that were "too risky" (too close to the skull). Since we started with a very coarse grid (`oct4`), losing these edge points makes the grid look sparse. In a real analysis (`oct6`), we start with `~8000` points, so this filtering is negligible.

In [ ]:
fwd_sol_fname = pathlib.Path('out_data') / 'sample_forward_solution_meg-fwd.fif'

mne.write_forward_solution(
    fname=fwd_sol_fname,
    fwd=fwd_solution,
    overwrite=True
)

## Compute noise covariance

As discussed, Noise Covariance is a mathematical model of the "background noise" in our sensors. We need this to tell the source localization algorithm what "silence" looks like, so it doesn't mistake sensor noise for brain activity.

Essentially, we answer the question: "How do the sensors fluctuate when the brain is doing nothing?"


###  Defining "Noise"
How do we capture pure noise?

1. **Empty Room Recording (MEG):** We often record the machine running in an empty room to capture environmental magnetic noise.
2. **Pre-Stimulus Baseline (EEG/MEG):** For evoked responses, we typically use the baseline period (e.g., -0.2s to 0.0s) before the stimulus appears. We assume the brain is "idle" relative to the stimulus during this window.

> **Note: EEG - The Average Reference**
>
> Unlike MEG, EEG measures voltage differences relative to a specific reference electrode (e.g., the nose or earlobe). This means one bad reference electrode can contaminate all your data.
Before computing covariance for source estimation, it is standard practice to apply an **"Average Reference"** (subtracting the mean of all sensors from every sensor). This spreads the error out and makes the physical model ("Forward Solution") much more stable.

### Computing and Visualizing
We use `mne.compute_covariance()` to calculate this matrix. One of the important params of this function is:
+ `rank=` - The "Rank" of the data tells us how many independent signals exist.
    - If you have 102 sensors, you theoretically have Rank 102.
    - However, if you applied 3 SSP Projectors (to remove heartbeats/blinks), you removed 3 dimensions of data.
    - Your new Rank will be $102 - 3 = 99$. MNE handles this automatically if you set    `rank='info'`, but it's good to verify visually.

To visualise the noise covariance, we can call `mne.viz.plot_cov()` which plots the *Covariance Matrix (Heatmap)* and *Eigenvalues (The Rank Check)*  for each channel type.

In [ ]:
noise_cov = mne.compute_covariance(
    epochs=epochs,
    tmax=0,       # Use the pre-stimulus period (Start -> 0s) as the noise baseline
    method=['shrunk', 'empirical'], # Robust methods for estimating covariance
    rank='info',  # Trust the info structure (which knows about the SSPs)
    n_jobs=-1
)

# Plot 1: Covariance Matrix & Eigenvalues
covariance_matrix_fig, eigen_values_fig = mne.viz.plot_cov(cov=noise_cov, info=epochs.info)

In [ ]:
covariance_matrix_fig

In [ ]:
eigen_values_fig

### Understanding the Plots:
+ **Covariance Matrix (Heatmap):** The diagonal is red (variance). The off-diagonal squares show correlations. Notice EEG has large red blocks (high correlation between neighbors) while Gradiometers are mostly white (low correlation).

+ **Eigenvalues (The Rank Check):** Look at the *Magnetometers* plot. The curve drops sharply to zero at index 99. This confirms that our 3 SSP projectors successfully removed 3 dimensions of noise ($102 - 3 = 99$).

### Whitening (The Quality Check)
Finally, we verify our noise model using "Whitening." The `.plot_white()` func divide our Evoked data by the Noise Covariance. If the noise model is perfect, during the baseline (pre-0s) the whitened signal should look like pure "Standard Normal Noise" (mean=0, std=1), i.e., in the channel plots the squiggly lines (trials) should wiggle between the red dashed lines (1.0, -1.0) during the baseline (left of 0.0s). For the Global Field Power (GFP) plot, if the blue line hovers on the red dashed line (1.0) during baseline, it's perfect; but if The blue line is way above or below the red line (meaning our noise model is too weak or too aggressive).

In [ ]:
epochs.average().plot_white(noise_cov=noise_cov)

## Create the Inverse Operator

As discussed, the **Inverse Operator** is the "Solver." It combines all the ingredients we have prepared so far - the head geometry (*Forward Solution*), the sensor noise (*Covariance*), and the sensor info into a single mathematical operator.

Once created, we can apply this operator to any data (*Evoked or Epochs*) to instantly estimate the source activity.

We use `mne.minimum_norm.make_inverse_operator()` to create the inverse operator. Two parameters are critical for tuning the physics:
1. `loose=0.2` *(Orientation Constraint):* Real neurons fire perpendicular to the cortex surface. However, our 3D model isn't perfect. Setting `loose=0.2` tells the solver: *"Assume the source is perpendicular, but allow about 20% variance (wiggle room) to account for model errors".*

2. `depth=0.8` *(Depth Weighting):* Signals from deep inside the brain are very weak by the time they reach the sensors. Without correction, the solver would bias everything to the surface. `depth=0.8` boosts the gain for deep sources so they have a fair chance of being detected.

In [ ]:
inverse_operator = mne.minimum_norm.make_inverse_operator(
    info=epochs.info,
    forward=fwd_solution,
    noise_cov=noise_cov,
    loose=0.2, # Allow some "wiggle room" for source orientation
    depth=0.8  # Boost deep sources so they aren't ignored
)

print(inverse_operator)

## Calculate the Source Estimate

Now to calculate the final **Source Estimate (STC)**, we call `mne.minimum_norm.apply_inverse()`. We will use this to apply our inverse operator to the *"Auditory/Left" evoked* data.

We need to specify a few key parameters:
1. `lambda2` *(Regularization): This controls how much we trust the data vs. the prior model. It is set inversely proportional to the Signal-to-Noise Ratio (SNR).
    + For averaged data (Evoked), a standard estimate is *SNR = 3.0* (signal is 3x stronger than noise).
    + Formula: `lambda2 = 1.0 / SNR**2`.

2. `method:` The mathematical algorithm used to solve the inverse problem.
    + `"MNE"` **(Minimum Norm Estimate)**: The classic solution. It minimizes the total energy. *Weakness:* It biases sources towards the surface (scalp).
    + `"dSPM"` **(Dynamic Statistical Parametric Mapping)**: Normalizes the MNE solution by the noise level. This creates a statistical map (Z-score) and fixes the surface bias. *Recommended for visualization*.
    + `"sLORETA"`: Another standardized method similar to *dSPM*, theoretically offering zero localization error in ideal conditions.
3. `pick_ori:` How to handle the vector direction.
    + `None:` Returns the magnitude (power) of the current at each vertex.

> Note: While the operator can be applied to raw data or single trials, we will apply it to our Evoked (averaged) data here to get a clean, low-noise estimate of the brain activity.

In [ ]:
stc = mne.minimum_norm.apply_inverse(
    evoked=epochs['Auditory/Left'].average(),
    inverse_operator=inverse_operator,
    lambda2=1.0 / 3.0 ** 2,  # 1.0 / SNR**2 (Regularization parameter)
    method='dSPM',           # Output will be statistical Z-scores (dimensionless)
    pick_ori=None
)

### Visualizing the Results

We can verify the result by plotting it on the 3D brain surface. We use the `.plot()` method, which opens an interactive 3D viewer.
+ `surface='inflated'`: Smoothed brain surface (removes deep wrinkles) so we can see activity inside the sulci.
+ `hemi='both'`: Show both left and right hemispheres.

In [ ]:
brain = stc.plot(
    surface="inflated", # Try "white" to see the realistic wrinkled cortex
    hemi="both",
    subjects_dir=subjects_dir,
    time_viewer=True    # Enables the slider to scroll through time
)

## Group Analysis
As discussed, we typically perform the source estimation pipeline for each participant individually. However, for the final scientific result, we need to compare or combine these results across the entire group.

### Sensor Level (Easy)
If we just want to look at the sensor data (ERP), we can simply average the Evoked objects from all subjects.
+ *Function:* `mne.grand_average([evoked1, evoked2, ...])`
+ *Result:* A single "Grand Average" ERP representing the group.

### Source Level (Harder - Morphing)
We cannot simply average source estimates because every person's brain is a different shape. Vertex #500 on Subject A might be the Visual Cortex, but Vertex #500 on Subject B might be the Parietal Lobe.

To solve this, we must **Morph** everyone's brain activity onto a standard template brain (typically `fsaverage` from FreeSurfer).

+ *The Process:* FreeSurfer "inflates" each subject's brain into a sphere and aligns the folding patterns (sulci and gyri) to the template sphere. This ensures that "Visual Cortex" activity from Subject A maps correctly to the "Visual Cortex" on the template.

+ *The Result:* If we have 20 participants, we end up with 20 stc files that all share the exact same grid (vertices). We can then perform statistical tests (t-tests, clusters) on these aligned maps.

```python
# --- Process Subject A ---
# Create a Morphing operator (from Subject A -> fsaverage)
morph_a = mne.compute_source_morph(
    stc_a, 
    subject_from='sample_a', 
    subject_to='fsaverage',  # The standard template brain
    subjects_dir=subjects_dir
) 
# Apply the morph to get the result on the standard brain
stc_fsaverage_a = morph_a.apply(stc_a)

# --- Process Subject B ---
# Create a Morphing operator (from Subject B -> fsaverage)
morph_b = mne.compute_source_morph(
    stc_b, 
    subject_from='sample_b',
    subject_to='fsaverage', # The standard template brain
    sibjects_dir=subjects_dir
)
# Apply the morph to get the result on the standard brain
stc_fsaverage_b = morph_b.apply(stc_b)

# --- Group Analysis (Hypothetical) ---
# Now that they are both on 'fsaverage', we can average them!
average_stc = (stc_fsaverage_a + stc_fsaverage_b) / 2
```

# The End & Have a great day!